# 03 In-Context Learning (ICL) & Vector Steering

**Real-World Scenario**: Dynamic Customer Support Ticket Sentiment & Intent Classifier.

This notebook demonstrates In-Context Learning (ICL) mechanisms, dynamic **k-NN demonstration selection** via vector embeddings, and **Ordering Bias Effects** (Recency Bias).

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from root .env file
load_dotenv()

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))


## Step 2: Dynamic k-NN Demonstration Selection (`FAISS` & `OpenAIEmbeddings`)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {"input": "My order #402 arrived damaged!", "output": "Urgent_Refund"},
    {"input": "When will my package be delivered?", "output": "Shipping_Status"},
    {"input": "Can I upgrade my subscription plan?", "output": "Billing_Upgrade"},
    {"input": "Screen is shattered on delivery.", "output": "Urgent_Refund"}
]

if os.getenv("OPENAI_API_KEY"):
    embeddings = OpenAIEmbeddings()
    texts = [ex["input"] for ex in examples]
    vectorstore = FAISS.from_texts(texts, embeddings, metadatas=examples)
    
    # Query: Retrieve top-1 semantically similar demonstration
    query = "Item arrived broken in box."
    results = vectorstore.similarity_search(query, k=1)
    selected_demo = results[0].metadata
    print("=== k-NN Selected Demonstration ===")
    print(f"Query: '{query}'")
    print(f"Selected Demo: Input: '{selected_demo['input']}' -> Output: '{selected_demo['output']}'")


## Step 3: Demonstrating Demonstration Ordering Bias (Recency Bias)

In [ ]:
from langchain_openai import ChatOpenAI

def test_ordering_bias(examples_order: list[dict], query: str):
    prompt_str = "Classify sentiment based on examples:
"
    for ex in examples_order:
        prompt_str += f"Text: {ex['text']} -> Sentiment: {ex['label']}
"
    prompt_str += f"Text: {query} -> Sentiment:"
    
    if os.getenv("OPENAI_API_KEY"):
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
        res = llm.invoke(prompt_str)
        print(f"Order: {[e['label'] for e in examples_order]} | Query Sentiment Output: {res.content.strip()}")

query_text = "The product works fine."
ex_pos = {"text": "Great service!", "label": "Positive"}
ex_neg = {"text": "Terrible quality.", "label": "Negative"}

print("=== Demonstration Ordering Bias Experiment ===")
test_ordering_bias([ex_pos, ex_neg], query_text)
test_ordering_bias([ex_neg, ex_pos], query_text) # Swapped order
